In [1]:

from urllib import request, error
from typing import List, Dict, Any, Iterable, Tuple, Optional
import os, json, time, hashlib, requests
from tqdm import tqdm

from pathlib import Path
import json, random, itertools, gc, statistics as st
import seqeval
import re
import requests
import time
from requests.exceptions import ReadTimeout, ConnectionError

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.cluster import KMeans
from IPython.display import clear_output

import torch
from transformers import (
    AutoTokenizer,
        AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,


    DataCollatorForTokenClassification
)

from seqeval.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
from seqeval.scheme import IOB2
from evaluate import load as load_metric
from tqdm.auto import tqdm

c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
HOST = "http://127.0.0.1:11434"

# >>> Escolha um que REALMENTE está instalado (copie/cole da sua /api/tags):
MODEL_KEY = "gpt-oss:20b"
# Alternativas que você tem: "qwen2.5:14b-instruct", "deepseek-r1:14b"

# Catálogo só para opções (não interfere no nome/tag do modelo)
OLLAMA_MODELS = {
    # --- já instalados ---
    "gemma2:9b":              {"options": {"num_ctx": 8000,  "temperature": 0.2}},
    "llama3.1:8b":            {"options": {"num_ctx": 8000,  "temperature": 0.2}},
    "qwen2.5:14b-instruct":   {"options": {"num_ctx": 12000, "temperature": 0.2}},
    "deepseek-r1:14b":        {"options": {"num_ctx": 8000,  "temperature": 0.2}},

    # --- Qwen mais leve ---
    "qwen2.5:7b":             {"options": {"num_ctx": 8000,  "temperature": 0.2}},
    "qwen2.5:7b-instruct":    {"options": {"num_ctx": 8000,  "temperature": 0.2}},  # use esta p/ chat

    # --- DeepSeek R1 mais leve ---
    "deepseek-r1:7b":         {"options": {"num_ctx": 8000,  "temperature": 0.2}},
    "deepseek-r1:1.5b":       {"options": {"num_ctx": 4096,  "temperature": 0.2}},  # ultraleve

    # --- “GPT OS” (gpt-oss open-weights) ---
    "gpt-oss:20b":            {"options": {"num_ctx": 8000,  "temperature": 0.2}},
}

In [3]:
SAFE_MODEL  = re.sub(r'[^A-Za-z0-9._-]+', '_', str(MODEL_KEY))  # "gemma2:9b" -> "gemma2_9b"

# --- pastas por modelo ---
BASE_DIR    = Path("caches") / SAFE_MODEL
DIR_CACHE   = BASE_DIR / "cache"
DIR_CKPT    = BASE_DIR / "checkpoints"
DIR_LOGS    = BASE_DIR / "logs"
for d in (DIR_CACHE, DIR_CKPT, DIR_LOGS):
    d.mkdir(parents=True, exist_ok=True)

In [4]:
CACHE_PATH  = DIR_CACHE / "ner_cache.jsonl" 

In [5]:
OLLAMA_MODELS[MODEL_KEY]["options"].update({
    "temperature": 0.0,
    "repeat_penalty": 1.1,
})

In [6]:
JSON_PATH = "../data/geocorpus-v2.json"        # ajuste se estiver noutra pasta
SEED_GLOBAL = 42
FEW_SHOT_K = 20

random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)

# ---------- ler o arquivo ----------
with open(JSON_PATH, encoding="utf-8") as f:
    raw = json.load(f)



In [7]:
records_geo = [
    {
        "sentence_id": i,
        "tokens"     : item["tokens"],
        "ner_tags"   : item["ner_tokens"],
    }
    for i, item in enumerate(raw)
]

geocorpus_full = Dataset.from_list(records_geo)

In [8]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({l for sent in geocorpus_full["ner_tags"] for l in sent})
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [9]:
iob_labels = (
    "O "
    "B-baciaSedimentar I-baciaSedimentar "
    "B-epoca I-epoca "
    "B-idade I-idade "
    "B-periodo I-periodo "
    "B-eon I-eon "
    "B-era I-era "
    "B-magmaticas I-magmaticas "
    "B-metamorficas I-metamorficas "
    "B-sedimentaresSiliciclasticas I-sedimentaresSiliciclasticas "
    "B-sedimentaresCarbonaticas I-sedimentaresCarbonaticas "
    "B-unidadeEstratigrafica I-unidadeEstratigrafica "
    "B-contextoGeologicoDeBacia I-contextoGeologicoDeBacia "
    "B-ambienteSedimentacao I-ambienteSedimentacao "
    "B-constituinteRochaSedimentar I-constituinteRochaSedimentar "
    "B-fosseis I-fosseis "
    "B-planctonico I-planctonico "
    "B-bentonico I-bentonico "
    "B-mineral I-mineral "
    "B-procedimentoMetodologico I-procedimentoMetodologico"
)

# Splits

In [10]:
def random_splits(
    ds: Dataset, test_size=0.2, seeds: List[int] = range(30)
) -> List[DatasetDict]:
    triples = []
    for s in seeds:
        train, dev = ds.train_test_split(test_size=test_size, seed=s).values()
        triples.append(DatasetDict(train=train, dev=dev))
    return triples

In [11]:
# def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
#     lengths = np.array([len(t) for t in ds["tokens"]])
#     thr = np.percentile(lengths, 100 * (1 - top_pct))
#     mask = lengths >= thr
#     return DatasetDict(train=ds.filter(~mask), dev=ds.filter(mask))


def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
    """20 % das sentenças mais longas viram conjunto de validação (dev)."""
    lengths = np.array([len(t) for t in ds["tokens"]])
    thr = np.percentile(lengths, 100 * (1 - top_pct))  
    mask = lengths >= thr  

    dev_idx = np.where(mask)[0].tolist()  # índices → list[int]
    train_idx = np.where(~mask)[0].tolist()

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Tamanho da sentenças


# def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
#     freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
#     rare = {w for w, c in freq.items() if c <= freq_thr}

#     def has_rare(example):
#         return any(w.lower() in rare for w in example["tokens"])

#     return DatasetDict(
#         train=ds.filter(lambda ex: not has_rare(ex)), dev=ds.filter(has_rare)
#     )


def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
    freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
    rare = {w for w, c in freq.items() if c <= freq_thr}

    keep_dev = []
    for sent in ds["tokens"]:
        print(sent)
        keep_dev.append(any(w.lower() in rare for w in sent))

    dev_idx = [i for i, x in enumerate(keep_dev) if x]
    train_idx = [i for i, x in enumerate(keep_dev) if not x]

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Raridade dos tokens

In [12]:
def split_adversarial(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeds = model.encode([" ".join(t) for t in ds["tokens"]], show_progress_bar=False)
    tree = BallTree(embeds, leaf_size=40)

    idx_train, idx_test = set(range(len(ds))), []
    # semente = ponto mais central
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    idx_train.remove(seed_idx)
    idx_test.append(seed_idx)
    print(k)
    while len(idx_test) < k:
        print(len(idx_test))
        dists, _ = tree.query(embeds[list(idx_train)], k=1, return_distance=True)
        nxt = list(idx_train)[int(np.argmax(dists))]
        idx_train.remove(nxt)
        idx_test.append(nxt)

    return DatasetDict(
        train=ds.select(sorted(idx_train)),
        dev=ds.select(sorted(idx_test)),
    )

    # Maximizando Wassertein Distance


def split_adversarial_fast(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    """
    Farthest-Point Sampling aproximando Wasserstein – versão vetorizada.
    Seleciona pct_test (~20 %) das sentenças como conjunto 'dev'.
    """
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    embeds = model.encode(
        [" ".join(t) for t in ds["tokens"]],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # acelera distância euclidiana ≈ cos
    )

    n = embeds.shape[0]
    idx_all = np.arange(n)

    # 1) ponto mais "central" (norma mais distante da média)
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    selected = [seed_idx]

    # 2) vetor de distâncias mínimas a qualquer ponto já escolhido
    min_dists = np.linalg.norm(embeds - embeds[seed_idx], axis=1)

    while len(selected) < k:
        next_idx = np.argmax(min_dists)
        selected.append(next_idx)

        # atualiza min_dists com a distância ao novo ponto — tudo de uma vez
        d_new = np.linalg.norm(embeds - embeds[next_idx], axis=1)
        min_dists = np.minimum(min_dists, d_new)

    train_idx = np.setdiff1d(idx_all, selected, assume_unique=True)

    return DatasetDict(
        train=ds.select(train_idx.tolist()),
        dev=ds.select(selected),
    )

In [13]:
# def loc_split(
#     dataset: Dataset, pct_test: float = 0.20, ngram: int = 4, seed: int = 42
# ) -> DatasetDict:
#     """
#     Split baseado em baixa sobreposição léxica (4-gram Jaccard).
#     Teste = pct_test das sentenças com menor overlap em relação ao pool.
#     """
#     # 1. Texto plano por sentença
#     docs = [" ".join(toks) for toks in dataset["tokens"]]

#     # 2. Vetorizar 4-grams (binário)
#     vect = CountVectorizer(
#         analyzer="word", ngram_range=(ngram, ngram), binary=True
#     ).fit(docs)
#     X = vect.transform(docs)

#     # 3. Similaridade Jaccard aproximada com matriz binária
#     # Jaccard(A,B) = |A∩B|/|A∪B| = 1 - |AΔB|/|A∪B|
#     # Usamos: overlap = (A·Bᵀ) / (|A|+|B|-A·Bᵀ)
#     bin_counts = X.sum(axis=1).A1

#     # Para cada doc i, escolhemos vizinho + próximo (fast):
#     from sklearn.metrics.pairwise import cosine_similarity

#     # (cosine no binário ∝ |A∩B|)
#     sim = cosine_similarity(X, dense_output=False)
#     # Soma dos top-k overlaps (k=5) como score
#     k = 5
#     topk = np.zeros(len(dataset))
#     for i in range(sim.shape[0]):
#         row = sim.getrow(i).toarray()[0]
#         idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
#         # overlap ≈ |∩|
#         inter = row[idx] * bin_counts[i]
#         uni = bin_counts[i] + bin_counts[idx] - inter
#         topk[i] = (inter / uni).mean()

#     # 4. Ordenar por overlap crescente ⇒ mais “novos” vão p/ teste
#     order = np.argsort(topk)
#     n_test = int(len(dataset) * pct_test)
#     test_idx = order[:n_test]
#     train_idx = order[n_test:]

#     return DatasetDict(
#         {"train": dataset.select(train_idx), "test": dataset.select(test_idx)}
#     )

In [14]:
def loc_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    ngram: int = 4,
    seed: int = 42,
) -> DatasetDict:
    """
    Divide por sobreposição léxica (n-gram Jaccard).
    Frações independentes para teste e validação.
    """
    docs = [" ".join(toks) for toks in dataset["tokens"]]

    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs)
    bin_counts = X.sum(axis=1).A1

    sim = cosine_similarity(X, dense_output=False)
    k = 5
    topk = np.zeros(len(dataset))
    for i in range(sim.shape[0]):
        row = sim.getrow(i).toarray()[0]
        idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
        inter = row[idx] * bin_counts[i]
        uni = bin_counts[i] + bin_counts[idx] - inter
        topk[i] = (inter / uni).mean()

    order = np.argsort(topk)  # baixo → alto overlap
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[:n_test]
    val_idx = order[n_test : n_test + n_val]
    train_idx = order[n_test + n_val :]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [15]:
def semantic_cluster_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    k: int | None = None,
    seed: int = 42,
) -> DatasetDict:
    """
    Clusters SBERT → reserva clusters distantes para test/val.
    """
    if k is None:
        k = int(np.sqrt(len(dataset)))

    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    embeddings = sbert.encode(
        [" ".join(t) for t in tqdm(dataset["tokens"])],
        batch_size=64,
        show_progress_bar=False,
    )

    km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(embeddings)
    labels = km.labels_
    centroids = km.cluster_centers_

    global_center = embeddings.mean(0, keepdims=True)
    dists = pairwise_distances(centroids, global_center).flatten()

    clusters_sorted = np.argsort(-dists)  # mais distantes primeiro
    test_clusters, val_clusters = set(), set()
    total_test = total_val = 0
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    for c in clusters_sorted:
        size = np.sum(labels == c)
        if total_test < n_test:  # preenche primeiro o teste
            test_clusters.add(c)
            total_test += size
        elif total_val < n_val:  # depois a validação
            val_clusters.add(c)
            total_val += size
        if total_test >= n_test and total_val >= n_val:
            break

    test_idx = np.where([lbl in test_clusters for lbl in labels])[0]
    val_idx = np.where([lbl in val_clusters for lbl in labels])[0]
    train_idx = np.where(
        [lbl not in test_clusters and lbl not in val_clusters for lbl in labels]
    )[0]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [16]:
def difficulty_scores(dataset: Dataset) -> np.ndarray:
    length = np.array([len(tok) for tok in dataset["tokens"]], dtype=float)
    length = (length - length.mean()) / length.std()

    dens = []
    for labels in dataset["ner_tags"]:
        non_o = sum(1 for l in labels if l != "O")
        dens.append(non_o / len(labels))
    dens = np.array(dens)
    dens = (dens - dens.mean()) / dens.std()

    ent_types, freq = [], Counter()
    for labels in dataset["ner_tags"]:
        types = [l[2:] for l in labels if l != "O"]
        ent_types.append(types[0] if types else "NONE")
    freq.update(ent_types)
    rarity = np.array([1 / freq[t] for t in ent_types])
    rarity = (rarity - rarity.mean()) / rarity.std()

    return length + dens + rarity


def reverse_curriculum_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    scores = difficulty_scores(dataset)
    order = np.argsort(scores)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[-n_test:]  # hardest
    val_idx = order[-(n_test + n_val) : -n_test]
    train_idx = order[: -(n_test + n_val)]

    rng = np.random.RandomState(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [17]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42) -> DatasetDict:
    """Testa só sentenças ≥ máx(len_train)."""
    sent_lens = np.array([len(t) for t in dataset["tokens"]])
    # separação inicial train/dev (randômica estratificando por tamanho grosso)
    idx_all   = np.arange(len(dataset))
    train_idx, temp_idx = train_test_split(idx_all,
                                           test_size=pct_test + pct_val,
                                           stratify=(sent_lens//5),  # bin len
                                           random_state=seed)
    # define longo-threshold como tamanho máx. do treino
    max_train_len = sent_lens[train_idx].max()
    # test = sentenças > threshold.  Caso falte/ sobre exemplos, ajusta.
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]
    # completa ou reduz para atingir pct_test
    need = int(pct_test*len(dataset)) - len(test_idx)
    if need > 0:
        test_idx.extend(resto_idx[:need])
        val_idx = resto_idx[need:]
    else:
        val_keep = int(pct_val*len(dataset))
        val_idx  = resto_idx[:val_keep]
        test_idx = test_idx[: int(pct_test*len(dataset))]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 2) Heuristic Rare-Words ----------------------------------
def heur_rare_split(dataset: Dataset,
                    pct_test: float = 0.20,
                    pct_val : float = 0.10,
                    seed: int = 42) -> DatasetDict:
    """Testa frases que contenham palavras do quintil + raro."""
    # contagem de frequência de token
    freqs = Counter(w for toks in dataset["tokens"] for w in toks)
    # define rareza: 20 % mais raras
    thresh = np.quantile(list(freqs.values()), 0.20)
    rare_set = {w for w,c in freqs.items() if c <= thresh}
    is_rare = np.array([any(w in rare_set for w in toks)
                        for toks in dataset["tokens"]])
    rare_idx   = np.where(is_rare)[0]
    common_idx = np.where(~is_rare)[0]
    # garante proporções desejadas
    n_test = int(pct_test*len(dataset))
    n_val  = int(pct_val *len(dataset))
    rng = np.random.default_rng(seed)
    test_idx = rng.choice(rare_idx, size=min(len(rare_idx), n_test),
                          replace=False)
    resto_idx = [i for i in rare_idx if i not in test_idx] + list(common_idx)
    val_idx  = rng.choice(resto_idx, size=n_val, replace=False)
    train_idx = [i for i in resto_idx if i not in val_idx]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 3) Standard (80-10-10) -----------------------------------
def std_split(dataset: Dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42) -> DatasetDict:
    """Split aleatório estratificado por comprimento (PTB-like)."""
    idx = np.arange(len(dataset))
    strat = (np.array([len(t) for t in dataset["tokens"]]) // 5)
    train_idx, temp_idx = train_test_split(idx, test_size=pct_test+pct_val,
                                          stratify=strat, random_state=seed)
    val_rel = pct_val / (pct_test+pct_val)
    val_idx, test_idx = train_test_split(temp_idx, test_size=1-val_rel,
                                         stratify=strat[temp_idx],
                                         random_state=seed)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 4) Adversarial (approx. Wasserstein) ---------------------
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Seleciona k frases 'mais distantes' recursivamente (BallTree + W₂)
    para compor o teste, lembrando Alg.-1 de Søgaard et al.【turn6file4】.
    """
    # SBERT embed (rápido na GPU / aceitável CPU)
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(toks) for toks in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    # BallTree para vizinhança eficiente
    tree = BallTree(emb, leaf_size=40)
    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)
    while len(test_idx) < int(pct_test*len(dataset)):
        # amostra candidata: ponto + longe do centro
        center = emb[list(idx_pool)].mean(0, keepdims=True)
        dists, _ = tree.query(center, k=len(idx_pool))
        farthest = list(idx_pool)[int(dists.argmax())]
        # pega-se farest e seus k-NN mais próximos  ⇒ aumenta diversidade
        nn = tree.query([emb[farthest]], k=k, return_distance=False)[0]
        for j in nn:
            if j in idx_pool and len(test_idx) < int(pct_test*len(dataset)):
                test_idx.append(j)
                idx_pool.remove(j)
    # retira val
    val_size = int(pct_val*len(dataset))
    val_idx  = rng.choice(list(idx_pool), size=val_size, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [18]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42,
                   bin_size: int = 10) -> DatasetDict:
    """
    Teste = sentenças mais longas que max(len(train)).
    Robusto a datasets pequenos: se estratificação falhar, usa split aleatório.
    """
    rng = np.random.default_rng(seed)
    idx_all   = np.arange(len(dataset))
    sent_lens = np.array([len(t) for t in dataset["tokens"]])

    # ------ 1) tenta split estratificado por baldes -------------------------
    strat = (sent_lens // bin_size)
    try:
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            stratify=strat,
            random_state=seed,
        )
    except ValueError:                       # classes com 1 amostra
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # ------ 2) escolhe test = len > max(train) ------------------------------
    max_train_len = sent_lens[train_idx].max()
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]

    # garante tamanhos exatos
    n_test_desired = int(pct_test * len(dataset))
    n_val_desired  = int(pct_val  * len(dataset))

    # completa teste se ficou pequeno
    if len(test_idx) < n_test_desired:
        extra = rng.choice(resto_idx,
                           size=n_test_desired - len(test_idx),
                           replace=False)
        test_idx.extend(extra)
        resto_idx = [i for i in resto_idx if i not in extra]

    # define validação
    val_idx  = rng.choice(resto_idx, size=n_val_desired, replace=False)
    train_idx = [i for i in idx_all if i not in test_idx and i not in val_idx]

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [19]:
def std_split(dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42,
              bin_size: int = 5) -> DatasetDict:
    """
    Split 80-10-10 robusto.
      • Tenta estratificar por comprimento // bin_size.
      • Se houver classes com <2 amostras, recua p/ split aleatório.
    """
    idx  = np.arange(len(dataset))
    bins = (np.array([len(t) for t in dataset["tokens"]]) // bin_size)

    try:
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            stratify=bins,
            random_state=seed,
        )
    except ValueError:                       # classes muito pequenas
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # fraciona temp em val / test mantendo proporção desejada
    val_share = pct_val / (pct_test + pct_val)
    try:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            stratify=bins[temp_idx],
            random_state=seed,
        )
    except ValueError:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [20]:
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Versão robusta: nunca “trava” antes de atingir n_test.
    Seleciona blocos de k sentenças mais distantes do centro iterativamente.
    """
    # -------------------------------- embeds -------------------------------
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(t) for t in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    tree = BallTree(emb, leaf_size=40)

    n_test = int(pct_test * len(dataset))
    n_val  = int(pct_val  * len(dataset))

    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)

    print(f"Selecionando {n_test} sentenças para teste…")
    step = 0
    while len(test_idx) < n_test and idx_pool:
        if step % 5 == 0:
            print(f"  {len(test_idx)} selecionadas…")
        step += 1

        pool_list = list(idx_pool)
        center = emb[pool_list].mean(0, keepdims=True)

        # distância euclidiana ao centro global restante
        dists, _ = tree.query(center, k=len(pool_list))
        farthest_local_idx = int(dists.argmax())
        farthest_global_idx = pool_list[farthest_local_idx]

        # número efetivo de vizinhos
        k_eff = min(k, len(idx_pool))
        nn = set(tree.query([emb[farthest_global_idx]],
                            k=k_eff,
                            return_distance=False)[0])

        # adiciona vizinhos ainda não selecionados
        for j in nn:
            if j in idx_pool and len(test_idx) < n_test:
                test_idx.append(j)
                idx_pool.remove(j)

        # Se nada foi adicionado (pode acontecer quando sobram <k únicos)
        if farthest_global_idx not in test_idx:
            test_idx.append(farthest_global_idx)
            idx_pool.remove(farthest_global_idx)

    # ------------------------ validação e treino ---------------------------
    val_idx = rng.choice(list(idx_pool), size=n_val, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [21]:
standard_split = std_split(geocorpus_full)
print('std')
# random_splt = random_splits(geocorpus_full)
# print('random')
heur_len = heur_len_split(geocorpus_full)
print("heur_len")
heur_rare = heur_rare_split(geocorpus_full)
print("heur_rare")
advers = adversarial_split(geocorpus_full)
print("advs")
loc = loc_split(geocorpus_full)
print("loc")
semantic = semantic_cluster_split(geocorpus_full)
print("semantic")
reverse = reverse_curriculum_split(geocorpus_full)
print("reverse")

std
heur_len
heur_rare
Selecionando 1054 sentenças para teste…
  0 selecionadas…
  24 selecionadas…
  47 selecionadas…
  71 selecionadas…
  96 selecionadas…
  121 selecionadas…
  144 selecionadas…
  165 selecionadas…
  187 selecionadas…
  211 selecionadas…
  232 selecionadas…
  251 selecionadas…
  271 selecionadas…
  288 selecionadas…
  305 selecionadas…
  329 selecionadas…
  349 selecionadas…
  364 selecionadas…
  383 selecionadas…
  399 selecionadas…
  417 selecionadas…
  433 selecionadas…
  452 selecionadas…
  470 selecionadas…
  488 selecionadas…
  501 selecionadas…
  515 selecionadas…
  529 selecionadas…
  542 selecionadas…
  554 selecionadas…
  565 selecionadas…
  577 selecionadas…
  590 selecionadas…
  597 selecionadas…
  610 selecionadas…
  628 selecionadas…
  642 selecionadas…
  658 selecionadas…
  671 selecionadas…
  685 selecionadas…
  698 selecionadas…
  713 selecionadas…
  724 selecionadas…
  738 selecionadas…
  754 selecionadas…
  771 selecionadas…
  785 selecionadas…
  7

100%|██████████| 5272/5272 [00:00<00:00, 1167064.48it/s]


semantic
reverse


# Pipeline Ollama

In [22]:
def _hash_key(tokens, model, shot_id="v1"):
    # Evita recomputar exemplos iguais (inclusive entre splits)
    s = json.dumps(tokens, ensure_ascii=False)
    return hashlib.md5(f"{model}|{shot_id}|{s}".encode()).hexdigest()

In [23]:
def _load_cache(path):
    cache = {}
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    cache[rec["k"]] = rec["v"]
                except Exception:
                    pass
    return cache

In [24]:
def _append_cache(path, k, v):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps({"k": k, "v": v}, ensure_ascii=False) + "\n")

In [25]:
def _normalize_ner_batch_json(obj):
    """
    Normaliza a saída do LLM para: [ {"id": int, "tags": [str, ...]}, ... ].
    Aceita variações comuns: objeto único, lista de objetos com "tokens",
    lista de listas, dict indexado por "0","1", etc.
    """
    if isinstance(obj, dict):
        if "id" in obj and ("tags" in obj or "tokens" in obj):
            tags = obj.get("tags") or obj.get("tokens")
            return [{"id": obj.get("id", 0), "tags": tags}]
        if all(isinstance(k, str) for k in obj.keys()):
            out = []
            try:
                for k in sorted(obj.keys(), key=lambda x: int(x)):
                    v = obj[k]
                    if isinstance(v, dict):
                        tags = v.get("tags") or v.get("tokens") or v.get("labels") or v
                    else:
                        tags = v
                    if isinstance(tags, list) and all(isinstance(t, str) for t in tags):
                        out.append({"id": int(k), "tags": tags})
                    else:
                        raise ValueError("Formato não reconhecido em item indexado.")
                return out
            except Exception:
                pass
        for key in ("results", "data", "output"):
            if key in obj:
                return _normalize_ner_batch_json(obj[key])
        raise ValueError(f"Formato JSON não reconhecido (dict): {obj}")

    if isinstance(obj, list):
        if not obj:
            return []
        if isinstance(obj[0], dict):
            norm = []
            for i, rec in enumerate(obj):
                tags = rec.get("tags") if "tags" in rec else rec.get("tokens") if "tokens" in rec else rec
                rid = rec.get("id", i)
                if isinstance(tags, list) and all(isinstance(t, str) for t in tags):
                    norm.append({"id": rid, "tags": tags})
                else:
                    if isinstance(rec, dict) and "tags" not in rec and "tokens" not in rec:
                        for key in ("tags", "tokens", "labels", "ner"):
                            if key in rec:
                                tags = rec[key]; break
                        if isinstance(tags, list) and all(isinstance(t, str) for t in tags):
                            norm.append({"id": rid, "tags": tags})
                        else:
                            raise ValueError(f"Item da lista sem tags reconhecíveis: {rec}")
                    else:
                        raise ValueError(f"Item da lista sem tags reconhecíveis: {rec}")
            return norm
        if isinstance(obj[0], list) and all(isinstance(t, str) for t in obj[0]):
            return [{"id": i, "tags": tags} for i, tags in enumerate(obj)]
        if all(isinstance(t, str) for t in obj):
            return [{"id": 0, "tags": obj}]
        raise ValueError(f"Formato JSON não reconhecido (list): {obj}")

    raise ValueError(f"Tipo JSON não reconhecido: {type(obj).__name__}")

In [26]:
def chunked(iterable, n):
    it = iter(iterable)
    while True:
        chunk = list(islice(it, n))
        if not chunk:
            return
        yield chunk

In [27]:
def tokens_to_text(tokens: List[str]) -> str:
    # Junta preservando espaços simples.
    return " ".join(tokens)


def fmt_example(ex: dict) -> str:
    toks = ex["tokens"]
    tags = ex.get("tags") or ex.get("ner_tags")  # compatível com seus dsets
    pairs_line = " ".join(f"{i+1}-{t}" for i, t in enumerate(tags))
    return f"Tokens: {' '.join(toks)}\nSaída: {pairs_line}"

def build_prompt(few_shot: List[Dict[str, Any]], query_tokens: List[str]) -> str:
    N = len(query_tokens)
    header = f"""
        Você é um anotador especialista em NER. Rotule **cada token** usando o esquema **IOB2**.

        Regras (case-sensitive):
        1) Use apenas os rótulos:\n{iob_labels}

        2) IOB2 (cheque silenciosamente):
        • Uma entidade inicia em B-<TIPO>.
        • I-<TIPO> só após B-<TIPO> ou I-<TIPO> do mesmo TIPO.
        • Nunca iniciar com I-.

        3) Formato ÚNICO de saída (sem texto extra):
        • Uma única linha com N pares índice-rótulo (1-based), separados por espaço.
        • Cada par: <índice>-<RÓTULO>.
        • Ex.: 1-O 2-B-magmaticas 3-I-magmaticas ... N-O

        4) Se houver N tokens, produza exatamente N rótulos.
        """.strip()

    examples = "\n\n### Exemplos\n" + "\n\n".join(fmt_example(ex) for ex in few_shot)

    task = f"""
        ### Tarefa
        Tokens: {tokens_to_text(query_tokens)}

        ### Saída esperada
        Retorne **apenas**: {{"tags": ["O", "B-...", "..."]}} com {N} itens.
        """.strip()

    return f"{header}\n\n{examples}\n\n{task}"


In [28]:
def select_few_shot(exemplars: Dataset, k=FEW_SHOT_K) -> list[dict]:
    # amostra k exemplos aleatórios (pode trocar por stratified sampling)
    return random.sample(list(exemplars), k)

In [29]:
def _format_few_shot_block(few_shot):
    """
    Aceita:
      - string pronta (usa como está, cortando em ~4000 chars), ou
      - lista/iterável de exemplos com campos 'tokens' e 'ner_tags'
    Retorna um bloco curto de few-shot em texto.
    """
    if isinstance(few_shot, str):
        return few_shot.strip()[:4000]

    parts = []
    try:
        for ex in few_shot:
            toks = ex.get("tokens") if isinstance(ex, dict) else ex["tokens"]
            tags = ex.get("ner_tags") if isinstance(ex, dict) else ex["ner_tags"]
            parts.append(json.dumps({"tokens": toks, "tags": tags}, ensure_ascii=False))
    except Exception:
        # fallback: forçar string
        return str(few_shot)[:4000]

    txt = "Exemplos rotulados (IOB2):\n" + "\n".join(parts)
    return txt[:4000]

In [30]:
# def _parse_json_loose(s: str):
#     dec = json.JSONDecoder()
#     i, n = 0, len(s)
#     vals = []
#     while i < n:
#         # pula espaços até encontrar um começo plausível
#         while i < n and s[i] not in "[{":
#             i += 1
#         if i >= n:
#             break
#         try:
#             v, j = dec.raw_decode(s, idx=i)
#             vals.append(v)
#             i = j
#         except json.JSONDecodeError:
#             i += 1
#     if not vals:
#         raise json.JSONDecodeError("no JSON found", s, 0)
#     return vals[0] if len(vals) == 1 else vals  # se vários, devolve lista

# class OllamaNER:
#     def __init__(self, model, few_shot_text, host="http://127.0.0.1:11434", keep_alive="30m", options=None, json_mode=True):
#         self.model = model
#         self.keep_alive = keep_alive
#         self.host = host
#         self.json_mode = json_mode
#         self.options = options or {
#             "temperature": 0,   # determinístico e ligeiramente mais rápido
#             # Dica: ajuste num_ctx para caber few-shot + entrada + saída com folga
#             "num_ctx": 6000,
#         }
#         self.sess = requests.Session()

#         few_shot_block = _format_few_shot_block(few_shot_text)

#         # ☑️ Envia o few-shot só uma vez (contexto “quase fixo”)
#         self._base_messages = [
#             {
#                 "role": "system",
#                 "content": (
#                     "Você é um rotulador NER. Dado um array de tokens, devolva rótulos IOB2 "
#                     """Regras (case-sensitive):
#                         1) Use apenas os rótulos:\n{iob_labels}

#                         2) IOB2 (cheque silenciosamente):
#                         • Uma entidade inicia em B-<TIPO>.
#                         • I-<TIPO> só após B-<TIPO> ou I-<TIPO> do mesmo TIPO.
#                         • Nunca iniciar com I-.

#                         3) Formato ÚNICO de saída (sem texto extra):
#                         • Uma única linha com N pares índice-rótulo (1-based), separados por espaço.
#                         • Cada par: <índice>-<RÓTULO>.
#                         • Ex.: 1-O 2-B-magmaticas 3-I-magmaticas ... N-O

#                         4) Se houver N palavras, produza exatamente N rótulos."""
#                     "EXCLUSIVAMENTE como JSON válido. NUNCA explique. Apenas JSON."
#                 ),
#             },
#             {"role": "user", "content": few_shot_block},
#             {"role": "assistant", "content": "OK"},
#         ]

#     def _chat_stream(self, user_content, connect_read_timeout=(5, 90)):
#         payload = {
#             "model": self.model,
#             "keep_alive": self.keep_alive,
#             "stream": True,
#             "messages": self._base_messages + [{"role": "user", "content": user_content}],
#             "options": self.options,
#         }
#         # ⚠️ não force format=json no streaming (alguns builds bufferizam)
#         with self.sess.post(
#             f"{self.host}/api/chat",
#             json=payload,
#             headers={"Accept": "text/event-stream"},
#             timeout=connect_read_timeout,
#             stream=True,
#         ) as r:
#             r.raise_for_status()
#             r.encoding = "utf-8"
#             chunks, got_any = [], False
#             start_t = time.time()

#             for line in r.iter_lines(decode_unicode=True, chunk_size=1):
#                 if not line:
#                     if time.time() - start_t > connect_read_timeout[1] - 1:
#                         break
#                     continue
#                 # aceita “data:{...}” (SSE) e linha pura “{...}” (NDJSON)
#                 payload_str = line[5:].strip() if line.startswith("data:") else line.strip()
#                 try:
#                     evt = json.loads(payload_str)
#                 except Exception:
#                     continue
#                 got_any = True
#                 part = evt.get("message", {}).get("content", "")
#                 if part:
#                     chunks.append(part)
#                 if evt.get("done"):
#                     break

#         raw = "".join(chunks).strip()
#         if raw.startswith("```"):
#             raw = re.sub(r"^```(?:json)?\s*", "", raw)
#             raw = re.sub(r"\s*```$", "", raw)
#         return raw if got_any else ""

#     # mantenha seu _normalize_ner_batch_json como está

#     def _normalize_ner_batch_json(self, obj):
#         if isinstance(obj, dict):
#             if "id" in obj and ("tags" in obj or "tokens" in obj):
#                 tags = obj.get("tags") or obj.get("tokens")
#                 return [{"id": obj.get("id", 0), "tags": tags}]
#             if all(isinstance(k, str) for k in obj.keys()):
#                 out = []
#                 try:
#                     for k in sorted(obj.keys(), key=lambda x: int(x)):
#                         v = obj[k]
#                         if isinstance(v, dict):
#                             tags = v.get("tags") or v.get("tokens") or v.get("labels") or v
#                         else:
#                             tags = v
#                         if isinstance(tags, list) and all(isinstance(t, str) for t in tags):
#                             out.append({"id": int(k), "tags": tags})
#                         else:
#                             raise ValueError("Formato não reconhecido em item indexado.")
#                     return out
#                 except Exception:
#                     pass
#             for key in ("results", "data", "output"):
#                 if key in obj:
#                     return self._normalize_ner_batch_json(obj[key])
#             raise ValueError(f"Formato JSON não reconhecido (dict): {obj}")

#         if isinstance(obj, list):
#             if not obj:
#                 return []
#             if isinstance(obj[0], dict):
#                 norm = []
#                 for i, rec in enumerate(obj):
#                     tags = rec.get("tags") if "tags" in rec else rec.get("tokens") if "tokens" in rec else rec
#                     rid = rec.get("id", i)
#                     if isinstance(tags, list) and all(isinstance(t, str) for t in tags):
#                         norm.append({"id": rid, "tags": tags})
#                     else:
#                         if isinstance(rec, dict) and "tags" not in rec and "tokens" not in rec:
#                             for key in ("tags", "tokens", "labels", "ner"):
#                                 if key in rec:
#                                     tags = rec[key]; break
#                             if isinstance(tags, list) and all(isinstance(t, str) for t in tags):
#                                 norm.append({"id": rid, "tags": tags})
#                             else:
#                                 raise ValueError(f"Item sem tags reconhecíveis: {rec}")
#                         else:
#                             raise ValueError(f"Item sem tags reconhecíveis: {rec}")
#                 return norm
#             if isinstance(obj[0], list) and all(isinstance(t, str) for t in obj[0]):
#                 return [{"id": i, "tags": tags} for i, tags in enumerate(obj)]
#             if all(isinstance(t, str) for t in obj):
#                 return [{"id": 0, "tags": obj}]
#             raise ValueError(f"Formato JSON não reconhecido (list): {obj}")

#         raise ValueError(f"Tipo JSON não reconhecido: {type(obj).__name__}")

#     def tag_batch(self, batch_tokens):
#         # 1) dimensiona num_predict para o lote atual (≈ 1 tag por token + folga)
#         expected = sum(len(t) for t in batch_tokens)
#         buffer   = 8 * max(1, len(batch_tokens))
#         opts = dict(self.options)
#         opts["num_predict"] = max(int(opts.get("num_predict", 0) or 0), expected + buffer)
#         self.options = opts  # simples e eficaz; se preferir, passe via payload local

#         inp = [{"id": i, "tokens": toks} for i, toks in enumerate(batch_tokens)]
#         prompt = (
#             "Rotule cada token no esquema IOB2. Use SOMENTE JSON no formato: "
#             "[ {\"id\": int, \"tags\": [\"O\"|\"B-...\"|\"I-...\"]}, ... ]\n"
#             "ENTRADA:\n" + json.dumps(inp, ensure_ascii=False)
#         )

#         # 2) tenta streaming; se vier vazio, faz fallback non-stream com format=json
#         try:
#             raw = self._chat_stream(prompt, connect_read_timeout=(5, 90))
#         except (ReadTimeout, ConnectionError):
#             raw = ""  # força fallback

#         if not raw:
#             # fallback sem stream, aqui pode forçar JSON completo
#             payload = {
#                 "model": self.model,
#                 "keep_alive": self.keep_alive,
#                 "stream": False,
#                 "messages": self._base_messages + [{"role": "user", "content": prompt}],
#                 "options": self.options,
#             }
#             if self.json_mode:
#                 payload["format"] = "json"
#             r = self.sess.post(f"{self.host}/api/chat", json=payload, timeout=(5, 600))
#             r.raise_for_status()
#             raw = r.json().get("message", {}).get("content", "") or ""

#         # 3) parse robusto
#         try:
#             parsed = json.loads(raw)
#         except Exception:
#             try:
#                 parsed = _parse_json_loose(raw)  # aceita NDJSON/valores múltiplos
#             except Exception:
#                 # Se ainda falhar, bisecta o batch (provável truncamento)
#                 if len(batch_tokens) > 1:
#                     mid = len(batch_tokens) // 2
#                     left  = self.tag_batch(batch_tokens[:mid])
#                     right = self.tag_batch(batch_tokens[mid:])
#                     return left + right
#                 # 1 exemplo só: tenta reemitir com num_predict maior e sem json_mode
#                 opts2 = dict(self.options); opts2["num_predict"] = max(opts2["num_predict"], expected + 128)
#                 jm = self.json_mode; self.json_mode = False
#                 try:
#                     raw2 = self._chat_stream(prompt, connect_read_timeout=(5, 120))
#                     parsed = json.loads(raw2) if raw2 else _parse_json_loose(raw2)
#                 finally:
#                     self.json_mode = jm

#         records = self._normalize_ner_batch_json(parsed)
#         out_sorted = sorted(records, key=lambda x: x["id"])
#         tags = [rec["tags"] for rec in out_sorted]

#         # 4) sanidade de comprimento
#         for i, (toks, tg) in enumerate(zip(batch_tokens, tags)):
#             if len(tg) != len(toks):
#                 tags[i] = (tg[: len(toks)]) if len(tg) > len(toks) else (tg + ["O"] * (len(toks) - len(tg)))
#         if len(tags) < len(batch_tokens):
#             for j in range(len(tags), len(batch_tokens)):
#                 tags.append(["O"] * len(batch_tokens[j]))
#         return tags

In [31]:
def _parse_json_loose(s: str):
    dec = json.JSONDecoder()
    i, n = 0, len(s)
    vals = []
    while i < n:
        # pula espaços até encontrar um começo plausível
        while i < n and s[i] not in "[{":
            i += 1
        if i >= n:
            break
        try:
            v, j = dec.raw_decode(s, idx=i)
            vals.append(v)
            i = j
        except json.JSONDecodeError:
            i += 1
    if not vals:
        raise json.JSONDecodeError("no JSON found", s, 0)
    return vals[0] if len(vals) == 1 else vals  # se vários, devolve lista

class OllamaNER:
    def __init__(self, model, few_shot_text, host="http://127.0.0.1:11434", keep_alive="30m", options=None, json_mode=True):
        self.model = model
        self.keep_alive = keep_alive
        self.host = host
        self.json_mode = json_mode
        self.options = options or {
            "temperature": 0,   # determinístico e ligeiramente mais rápido
            # Dica: ajuste num_ctx para caber few-shot + entrada + saída com folga
            "num_ctx": 6000,
        }
        self.sess = requests.Session()

        few_shot_block = _format_few_shot_block(few_shot_text)

        # ☑️ Envia o few-shot só uma vez (contexto “quase fixo”)
        self._base_messages = [
            {
                "role": "system",
                "content": (
                    "Você é um rotulador NER. Dado um array de tokens, devolva rótulos IOB2 "
                    """Regras (case-sensitive):
                        1) Use apenas os rótulos:\n{iob_labels}

                        2) IOB2 (cheque silenciosamente):
                        • Uma entidade inicia em B-<TIPO>.
                        • I-<TIPO> só após B-<TIPO> ou I-<TIPO> do mesmo TIPO.
                        • Nunca iniciar com I-.

                        3) Formato ÚNICO de saída (sem texto extra):
                        • Uma única linha com N pares índice-rótulo (1-based), separados por espaço.
                        • Cada par: <índice>-<RÓTULO>.
                        • Ex.: 1-O 2-B-magmaticas 3-I-magmaticas ... N-O

                        4) Se houver N palavras, produza exatamente N rótulos."""
                    "EXCLUSIVAMENTE como JSON válido. NUNCA explique. Apenas JSON."
                ),
            },
            {"role": "user", "content": few_shot_block},
            {"role": "assistant", "content": "OK"},
        ]

    def _chat_stream(self, user_content, connect_read_timeout=(5, 90)):
        payload = {
            "model": self.model,
            "keep_alive": self.keep_alive,
            "stream": True,
            "messages": self._base_messages + [{"role": "user", "content": user_content}],
            "options": self.options,
        }
        # ⚠️ não force format=json no streaming (alguns builds bufferizam)
        with self.sess.post(
            f"{self.host}/api/chat",
            json=payload,
            headers={"Accept": "text/event-stream"},
            timeout=connect_read_timeout,
            stream=True,
        ) as r:
            r.raise_for_status()
            r.encoding = "utf-8"
            chunks, got_any = [], False
            start_t = time.time()

            for line in r.iter_lines(decode_unicode=True, chunk_size=1):
                if not line:
                    if time.time() - start_t > connect_read_timeout[1] - 1:
                        break
                    continue
                # aceita “data:{...}” (SSE) e linha pura “{...}” (NDJSON)
                payload_str = line[5:].strip() if line.startswith("data:") else line.strip()
                try:
                    evt = json.loads(payload_str)
                except Exception:
                    continue
                got_any = True
                part = evt.get("message", {}).get("content", "")
                if part:
                    chunks.append(part)
                if evt.get("done"):
                    break

        raw = "".join(chunks).strip()
        if raw.startswith("```"):
            raw = re.sub(r"^```(?:json)?\s*", "", raw)
            raw = re.sub(r"\s*```$", "", raw)
        return raw if got_any else ""

    # mantenha seu _normalize_ner_batch_json como está

    def _normalize_ner_batch_json(self, obj):
        if isinstance(obj, dict):
            if "id" in obj and ("tags" in obj or "tokens" in obj):
                tags = obj.get("tags") or obj.get("tokens")
                return [{"id": obj.get("id", 0), "tags": _coerce_tags_list(tags)}]
            if all(isinstance(k, str) for k in obj.keys()):
                out = []
                try:
                    for k in sorted(obj.keys(), key=lambda x: int(x)):
                        v = obj[k]
                        if isinstance(v, dict):
                            tags = v.get("tags") or v.get("tokens") or v.get("labels") or v
                        else:
                            tags = v
                        if isinstance(tags, list) and all(isinstance(t, str) for t in tags):
                            out.append({"id": int(k), "tags": tags})
                        else:
                            raise ValueError("Formato não reconhecido em item indexado.")
                    return out
                except Exception:
                    pass
            for key in ("results", "data", "output"):
                if key in obj:
                    return self._normalize_ner_batch_json(obj[key])
            raise ValueError(f"Formato JSON não reconhecido (dict): {obj}")

        if isinstance(obj, list):
            if not obj:
                return []
            if isinstance(obj[0], dict):
                norm = []
                for i, rec in enumerate(obj):
                    tags = rec.get("tags") if "tags" in rec else rec.get("tokens") if "tokens" in rec else rec
                    rid = rec.get("id", i)
                    tags = _coerce_tags_list(tags)
                    if isinstance(tags, list) and all(isinstance(t, str) for t in tags):
                        norm.append({"id": rid, "tags": tags})
                    else:
                        if isinstance(rec, dict) and "tags" not in rec and "tokens" not in rec:
                            for key in ("tags", "tokens", "labels", "ner"):
                                if key in rec:
                                    tags = rec[key]; break
                            if isinstance(tags, list) and all(isinstance(t, str) for t in tags):
                                norm.append({"id": rid, "tags": tags})
                            else:
                                raise ValueError(f"Item sem tags reconhecíveis: {rec}")
                        else:
                            raise ValueError(f"Item sem tags reconhecíveis: {rec}")
                return norm
            if isinstance(obj[0], list) and all(isinstance(t, str) for t in obj[0]):
                return [{"id": i, "tags": tags} for i, tags in enumerate(obj)]
            if all(isinstance(t, str) for t in obj):
                return [{"id": 0, "tags": _coerce_tags_list(obj)}] 
            raise ValueError(f"Formato JSON não reconhecido (list): {obj}")

        raise ValueError(f"Tipo JSON não reconhecido: {type(obj).__name__}")

    def tag_batch(self, batch_tokens):
        # 1) dimensiona num_predict para o lote atual (≈ 1 tag por token + folga)
        expected = sum(len(t) for t in batch_tokens)
        buffer   = 8 * max(1, len(batch_tokens))
        opts = dict(self.options)
        opts["num_predict"] = max(int(opts.get("num_predict", 0) or 0), expected + buffer)
        self.options = opts  # simples e eficaz; se preferir, passe via payload local

        inp = [{"id": i, "tokens": toks} for i, toks in enumerate(batch_tokens)]
        prompt = (
            "Rotule cada token no esquema IOB2. Use SOMENTE JSON no formato: "
            "[ {\"id\": int, \"tags\": [\"O\"|\"B-...\"|\"I-...\"]}, ... ]\n"
            "ENTRADA:\n" + json.dumps(inp, ensure_ascii=False)
        )

        # 2) tenta streaming; se vier vazio, faz fallback non-stream com format=json
        try:
            raw = self._chat_stream(prompt, connect_read_timeout=(5, 90))
        except (ReadTimeout, ConnectionError):
            raw = ""  # força fallback

        if not raw:
            # fallback sem stream, aqui pode forçar JSON completo
            payload = {
                "model": self.model,
                "keep_alive": self.keep_alive,
                "stream": False,
                "messages": self._base_messages + [{"role": "user", "content": prompt}],
                "options": self.options,
            }
            if self.json_mode:
                payload["format"] = "json"
            r = self.sess.post(f"{self.host}/api/chat", json=payload, timeout=(5, 600))
            r.raise_for_status()
            raw = r.json().get("message", {}).get("content", "") or ""

        # 3) parse robusto
        try:
            parsed = json.loads(raw)
        except Exception:
            try:
                parsed = _parse_json_loose(raw)  # seu parser “solto” que aceita NDJSON/múltiplos valores
            except Exception:
                parsed = None

        # 3.1) Se parseou mas NÃO tem 'tags' (eco da entrada ou formato errado), re-pergunte sem stream
        if (parsed is None) or _looks_like_echo(parsed) or not _has_tags_struct(parsed):
            # prompt de reparo: enfatiza que NÃO deve repetir a entrada nem usar chave 'tokens'
            repair_prompt = (
                "NÃO REPITA A ENTRADA. Devolva SOMENTE JSON no formato exato: "
                "[{\"id\": int, \"tags\": [\"O\"|\"B-...\"|\"I-...\"]}, ...]\n"
                "Não inclua a chave 'tokens'.\n"
                "ENTRADA:\n" + json.dumps(inp, ensure_ascii=False)
            )
            # non-stream + format=json para forçar JSON completo
            payload = {
                "model": self.model,
                "keep_alive": self.keep_alive,
                "stream": False,
                "messages": self._base_messages + [{"role": "user", "content": repair_prompt}],
                "options": self.options,
            }
            if self.json_mode:
                payload["format"] = "json"

            r = self.sess.post(f"{self.host}/api/chat", json=payload, timeout=(5, 600))
            r.raise_for_status()
            raw2 = r.json().get("message", {}).get("content", "") or ""

            try:
                parsed2 = json.loads(raw2)
            except Exception:
                try:
                    parsed2 = _parse_json_loose(raw2)
                except Exception:
                    parsed2 = None

            if (parsed2 is None) or _looks_like_echo(parsed2) or not _has_tags_struct(parsed2):
                # ainda ruim → bisecta o batch (isola o exemplo problemático)
                if len(batch_tokens) > 1:
                    mid = len(batch_tokens) // 2
                    left  = self.tag_batch(batch_tokens[:mid])
                    right = self.tag_batch(batch_tokens[mid:])
                    return left + right
                # 1 único exemplo e ainda eco? decida política:
                #  a) levantar erro:
                raise BadModelOutput("Modelo não retornou 'tags'; ecoou 'tokens'.")
                #  b) (opcional) fallback benigno: tudo 'O'
                # return [["O"] * len(batch_tokens[0])]

            parsed = parsed2  # ok, seguimos com o reparo

        records = self._normalize_ner_batch_json(parsed)
        out_sorted = sorted(records, key=lambda x: x["id"])
        tags = [rec["tags"] for rec in out_sorted]

        # sanidade de comprimento
        for i, (toks, tg) in enumerate(zip(batch_tokens, tags)):
            if len(tg) != len(toks):
                tags[i] = (tg[: len(toks)]) if len(tg) > len(toks) else (tg + ["O"] * (len(toks) - len(tg)))
        if len(tags) < len(batch_tokens):
            for j in range(len(tags), len(batch_tokens)):
                tags.append(["O"] * len(batch_tokens[j]))
        return tags

In [32]:
# class OllamaNER:
#     def __init__(self, model, few_shot_text, host="http://127.0.0.1:11434", keep_alive="30m", options=None, json_mode=True):
#         self.model = model
#         self.keep_alive = keep_alive
#         self.host = host
#         self.json_mode = json_mode
#         self.options = options or {
#             "temperature": 0,   # determinístico e ligeiramente mais rápido
#             # Dica: ajuste num_ctx para caber few-shot + entrada + saída com folga
#             "num_ctx": 6000,
#         }
#         self.sess = requests.Session()

#         few_shot_block = _format_few_shot_block(few_shot_text)

#         # ☑️ Envia o few-shot só uma vez (contexto “quase fixo”)
#         self._base_messages = [
#             {
#                 "role": "system",
#                 "content": (
#                     "Você é um rotulador NER. Dado um array de tokens, devolva rótulos IOB2 "
#                     """Regras (case-sensitive):
#                         1) Use apenas os rótulos:\n{iob_labels}

#                         2) IOB2 (cheque silenciosamente):
#                         • Uma entidade inicia em B-<TIPO>.
#                         • I-<TIPO> só após B-<TIPO> ou I-<TIPO> do mesmo TIPO.
#                         • Nunca iniciar com I-.

#                         3) Formato ÚNICO de saída (sem texto extra):
#                         • Uma única linha com N pares índice-rótulo (1-based), separados por espaço.
#                         • Cada par: <índice>-<RÓTULO>.
#                         • Ex.: 1-O 2-B-magmaticas 3-I-magmaticas ... N-O

#                         4) Se houver N palavras, produza exatamente N rótulos."""
#                     "EXCLUSIVAMENTE como JSON válido. NUNCA explique. Apenas JSON."
#                 ),
#             },
#             {"role": "user", "content": few_shot_block},
#             {"role": "assistant", "content": "OK"},
#         ]

    
#     def _normalize_ner_batch_json(self, obj):
#         if isinstance(obj, dict):
#             if "id" in obj and ("tags" in obj or "tokens" in obj):
#                 tags = obj.get("tags") or obj.get("tokens")
#                 return [{"id": obj.get("id", 0), "tags": tags}]
#             if all(isinstance(k, str) for k in obj.keys()):
#                 out = []
#                 try:
#                     for k in sorted(obj.keys(), key=lambda x: int(x)):
#                         v = obj[k]
#                         if isinstance(v, dict):
#                             tags = v.get("tags") or v.get("tokens") or v.get("labels") or v
#                         else:
#                             tags = v
#                         if isinstance(tags, list) and all(isinstance(t, str) for t in tags):
#                             out.append({"id": int(k), "tags": tags})
#                         else:
#                             raise ValueError("Formato não reconhecido em item indexado.")
#                     return out
#                 except Exception:
#                     pass
#             for key in ("results", "data", "output"):
#                 if key in obj:
#                     return self._normalize_ner_batch_json(obj[key])
#             raise ValueError(f"Formato JSON não reconhecido (dict): {obj}")

#         if isinstance(obj, list):
#             if not obj:
#                 return []
#             if isinstance(obj[0], dict):
#                 norm = []
#                 for i, rec in enumerate(obj):
#                     tags = rec.get("tags") if "tags" in rec else rec.get("tokens") if "tokens" in rec else rec
#                     rid = rec.get("id", i)
#                     if isinstance(tags, list) and all(isinstance(t, str) for t in tags):
#                         norm.append({"id": rid, "tags": tags})
#                     else:
#                         if isinstance(rec, dict) and "tags" not in rec and "tokens" not in rec:
#                             for key in ("tags", "tokens", "labels", "ner"):
#                                 if key in rec:
#                                     tags = rec[key]; break
#                             if isinstance(tags, list) and all(isinstance(t, str) for t in tags):
#                                 norm.append({"id": rid, "tags": tags})
#                             else:
#                                 raise ValueError(f"Item sem tags reconhecíveis: {rec}")
#                         else:
#                             raise ValueError(f"Item sem tags reconhecíveis: {rec}")
#                 return norm
#             if isinstance(obj[0], list) and all(isinstance(t, str) for t in obj[0]):
#                 return [{"id": i, "tags": tags} for i, tags in enumerate(obj)]
#             if all(isinstance(t, str) for t in obj):
#                 return [{"id": 0, "tags": obj}]
#             raise ValueError(f"Formato JSON não reconhecido (list): {obj}")

#         raise ValueError(f"Tipo JSON não reconhecido: {type(obj).__name__}")


#     def _chat_stream(self, user_content, connect_read_timeout=(5, 600)):
#         payload = {
#             "model": self.model,
#             "keep_alive": self.keep_alive,
#             "stream": True,  # streaming = evita ReadTimeout e evita r.json()
#             "messages": self._base_messages + [{"role": "user", "content": user_content}],
#             "options": self.options,
#         }


#         if self.json_mode:
#             # Obriga JSON “puro” (reduz pós-processamento e tokens gerados)
#             payload["format"] = "json"

#         with self.sess.post(
#         f"{self.host}/api/chat",
#         json=payload,
#         headers={"Accept": "text/event-stream"},
#         timeout=connect_read_timeout,  # (connect, read)
#         stream=True
#         ) as r:
#             r.raise_for_status()
#             r.encoding = "utf-8"  # garante str no iter_lines
#             chunks = []
#             got_any = False
#             start_t = time.time()

#             for line in r.iter_lines(decode_unicode=True, chunk_size=1):
#                 if not line:
#                     if time.time() - start_t > connect_read_timeout[1] - 1:
#                         break
#                     continue

#                 # Aceita SSE (data: {...}) e NDJSON (linha é {...})
#                 payload_str = line[5:].strip() if line.startswith("data:") else line.strip()

#                 try:
#                     evt = json.loads(payload_str)
#                 except Exception:
#                     # debug opcional:
#                     # print("linha não-JSON:", repr(line[:120]))
#                     continue

#                 got_any = True
#                 part = evt.get("message", {}).get("content", "")
#                 if part:
#                     chunks.append(part)
#                 if evt.get("done"):
#                     break

#             raw = "".join(chunks).strip()
#             # limpa cercas ```json ... ```
#             if raw.startswith("```"):
#                 raw = re.sub(r"^```(?:json)?\s*", "", raw)
#                 raw = re.sub(r"\s*```$", "", raw)

#             # se nada chegou pelo stream, devolva "" para acionar fallback non-stream
#             return raw if got_any else ""

#     def tag_batch(self, batch_tokens):
#         """
#         batch_tokens: List[List[str]] (vários exemplos em uma chamada)
#         Retorna: List[List[str]] na mesma ordem.
#         """
#         # ⚠️ Pedimos uma lista de objetos { "id": int, "tags": [...] } para manter ordenação
#         inp = [{"id": i, "tokens": toks} for i, toks in enumerate(batch_tokens)]
#         prompt = (
#             "Rotule cada token no esquema IOB2. Rótulos permitidos já foram definidos no few-shot.\n"
#             "ENTRADA:\n" + json.dumps(inp, ensure_ascii=False) + "\n\n"
#             "SAÍDA (JSON ESTRITO):\n"
#             "[ {\"id\": int, \"tags\": [\"O\"|\"B-...\"|\"I-...\"]}, ... ]\n"
#         )

#         try:
#             raw = self._chat_stream(prompt, connect_read_timeout=(5, 600))
#         except (ReadTimeout, ConnectionError):
#             if len(batch_tokens) == 1:
#                 raw = self._chat_stream(prompt, connect_read_timeout=(5, 1200))
#             else:
#                 mid = len(batch_tokens) // 2
#                 left  = self.tag_batch(batch_tokens[:mid])
#                 right = self.tag_batch(batch_tokens[mid:])
#                 return left + right

#         # --- parsing robusto do JSON ---
#         try:
#             out = json.loads(raw)
#         except Exception:
#             # Reparo simples: pega o primeiro '[' até o último ']'
#             start, end = raw.find("["), raw.rfind("]") + 1
#             if start != -1 and end > start:
#                 out = json.loads(raw[start:end])
#             else:
#                 start, end = raw.find("{"), raw.rfind("}") + 1
#                 out = json.loads(raw[start:end])

#         records = self._normalize_ner_batch_json(out)
#         out_sorted = sorted(records, key=lambda x: x["id"])
#         tags = [rec["tags"] for rec in out_sorted]

#         # Segurança: garante mesmo tamanho de cada exemplo
#         for i, (toks, tg) in enumerate(zip(batch_tokens, tags)):
#             if len(tg) != len(toks):
#                 # Heurística: corta ou preenche com 'O'
#                 if len(tg) > len(toks):
#                     tags[i] = tg[: len(toks)]
#                 else:
#                     tags[i] = tg + ["O"] * (len(toks) - len(tg))
#         return tags

In [33]:
def ollama_generate(
    model: str,
    prompt: str,
    options: Optional[Dict[str, Any]] = None,
    host: str = "http://localhost:11434",
    timeout: int = 120,
    stream: bool = False,
) -> str:
    """
    Wrapper do endpoint /api/generate.
    """
    url = f"{host}/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": stream,
        "options": options or {},
        # "stop": ["\n\n"],  # ajuste opcional se algum modelo insistir em linhas extras
    }
    # Timeout levemente alto pois alguns modelos podem demorar.
    resp = requests.post(url, json=payload, timeout=timeout)
    resp.raise_for_status()

    if stream:
        # Se usar stream=True, agregamos as linhas 'data: {...}'
        text = ""
        for line in resp.iter_lines():
            if not line:
                continue
            try:
                obj = json.loads(line.decode("utf-8"))
            except Exception:
                continue
            text += obj.get("response", "")
        return text
    else:
        data = resp.json()
        return data.get("response", "")

In [34]:
def predict_labels_ollama(few_shot, query_tokens, model_key=MODEL_KEY, host=HOST):
    cfg = OLLAMA_MODELS[model_key]
    prompt = build_prompt(few_shot, query_tokens)
    print(prompt)
    raw = ollama_generate(model=model_key, prompt=prompt, options=cfg.get("options"), host=host, stream=False)
    return raw

In [35]:
SESSION = requests.Session()

In [36]:
def ollama_generate_json(model: str, prompt: str, host="http://127.0.0.1:11434",
                         num_predict:int=64, num_ctx:int=2048, timeout=(10, 600)):
    payload = {
        "model": model,
        "prompt": prompt,
        "format": "json",     # resposta vem como string JSON em data['response']
        "stream": False,      # evita travar se o cliente não consumir stream
        "keep_alive": "30m",
        "options": {
            "temperature": 0,
            "seed": 42,
            "num_ctx": num_ctx*30,
            "num_predict": max(64, 5 * num_predict)
        }
    }
    return SESSION.post(f"{host}/api/generate", json=payload, timeout=timeout)

In [37]:
def predict_labels_ollama(tokens: List[str],
                          few_shot_text: str = "",
                          model: str = "llama3.1:8b") -> List[str]:
    # prompt curto e objetivo; evite few-shots gigantes (cole 1–2 exemplos no few_shot_text)
    
    N = len(tokens)
    prompt = build_prompt(few_shot_text, tokens)
    # retry leve para 5xx/timeout
    for attempt in range(3):
        try:
            r = ollama_generate_json(
                model=model,
                prompt=prompt,
                # use o número de tokens como base para calcular num_predict interno
                num_predict=min(64, 3 * len(tokens)),
                num_ctx=N
            )
            r.raise_for_status()
            data = r.json()
            attempt = 3
            return data['response']
        except Exception:
            if attempt == 2:
                raise
            time.sleep(1.5 * (attempt + 1))

In [38]:
class BadModelOutput(Exception):
    pass

def _has_tags_struct(x):
    try:
        items = x if isinstance(x, list) else [x]
        for it in items:
            if isinstance(it, list):
                # lista de registros
                for r in it:
                    if isinstance(r, dict) and isinstance(r.get("tags"), list):
                        return True
            elif isinstance(it, dict) and isinstance(it.get("tags"), list):
                return True
        return False
    except Exception:
        return False

def _looks_like_echo(x):
    # lista (ou lista de listas) de dicts com 'tokens' e sem 'tags'
    try:
        if isinstance(x, list) and x:
            first = x[0]
            if isinstance(first, list) and first and isinstance(first[0], dict):
                return all(("tokens" in d and "tags" not in d) for d in first)
            if isinstance(first, dict):
                return ("tokens" in first and "tags" not in first)
        if isinstance(x, dict):
            return ("tokens" in x and "tags" not in x)
    except Exception:
        return False
    return False

In [39]:
def _coerce_tag_value(d):
    # tenta chaves comuns de IOB2
    for key in ("tag", "label", "iob", "iob_tag", "ner", "entity", "type"):
        v = d.get(key)
        if isinstance(v, str):
            v = v.strip()
            # aceita O, B-..., I-...
            if v == "O" or re.match(r"^[BI]-", v):
                return v
    # se nada servir, marca como O
    return "O"

def _coerce_tags_list(x):
    """
    x: pode ser lista de strings OU lista de dicts (word/type, etc.)
    retorna sempre lista de strings (IOB2-like), substituindo inválidos por 'O'
    """
    if not isinstance(x, list):
        return []
    if all(isinstance(t, str) for t in x):
        return x
    if all(isinstance(t, dict) for t in x):
        return [_coerce_tag_value(t) for t in x]
    # mix estranho: tenta extrair, senão devolve tudo 'O'
    out = []
    for t in x:
        if isinstance(t, str):
            out.append(t if (t == "O" or re.match(r"^[BI]-", t or "")) else "O")
        elif isinstance(t, dict):
            out.append(_coerce_tag_value(t))
        else:
            out.append("O")
    return out

In [40]:
FAILED_LOG  = DIR_LOGS  / "ner_failed_indices.log" 

def log_fail(split_name, idx, reason):
    with open(FAILED_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps({"split": split_name, "idx": idx, "err": str(reason)}) + "\n")

def get_checkpoint_path(nome: str) -> Path:
    """Caminho do checkpoint do split 'nome', por modelo."""
    return DIR_CKPT / f"{SAFE_MODEL}__{nome}.ckpt.json"

def load_checkpoint(nome):
    p = get_checkpoint_path(nome)
    if os.path.exists(p):
        try:
            return int(json.load(open(p, "r", encoding="utf-8")).get("next_idx", 0))
        except Exception:
            return 0
    return 0

def save_checkpoint(nome, next_idx):
    with open(get_checkpoint_path(nome), "w", encoding="utf-8") as f:
        json.dump({"next_idx": int(next_idx)}, f)

# Prompts

In [41]:
# few_shot = [
#     {"tokens": ["o", "maj", "é", "composto"], "tags": ["O","O","O","O"]},
#     {"tokens": ["dioritos", "e", "monzodioritos"], "tags": ["B-magmaticas","O","I-magmaticas"]},
# ]
# query_tokens = ["o","maj","é","composto","por","monzodioritos","."]
# preds = predict_labels_ollama(few_shot, query_tokens)  # usa MODEL_KEY
# preds

In [42]:
splits = [standard_split, heur_len, heur_rare, advers]

In [43]:
splits_names = ['standard', 'heur_len', 'heur_rare', 'adversarial']

In [44]:
MODEL_KEY

'gpt-oss:20b'

In [45]:
model_escolhido = MODEL_KEY

In [46]:
def run_fast_inference(
    splits, splits_names, model_escolhido, FEW_SHOT_K, select_few_shot, SEED_GLOBAL
):
    random.seed(SEED_GLOBAL)

    golds, predictions = {}, {}
    cache = _load_cache(CACHE_PATH)

    for split, nome in zip(splits, splits_names):
        split_base = split
        total = min(500, len(split_base["test"]))  # ajuste se quiser rodar tudo
        few_shot = select_few_shot(split_base["train"], FEW_SHOT_K)

        ner = OllamaNER(
            model=model_escolhido,
            few_shot_text=few_shot,
            options={
                "temperature": 0,
                "num_ctx": 768,
                "num_predict": 1536,
            },
            keep_alive="10m",
            json_mode=True,
        )

        all_test = list(split_base["test"].select(range(total)))
        split_salt = f"split:{nome}"
        BATCH_SIZE = 8

        # Pré-aloca
        golds[nome] = [None] * total
        predictions[nome] = [None] * total

        # Retomada: onde paramos?
        start_idx = load_checkpoint(nome)

        # Preenche rapidamente os índices anteriores usando o cache (se existir)
        if start_idx > 0:
            for ex_idx in range(0, min(start_idx, total)):
                ex = all_test[ex_idx]
                golds[nome][ex_idx] = [ex["ner_tags"]]
                k = _hash_key(ex["tokens"], model_escolhido, shot_id=split_salt)
                if k in cache and predictions[nome][ex_idx] is None:
                    predictions[nome][ex_idx] = [cache[k]]
            # Garantia: se algum anterior continuar None, deixamos p/ reprocessar abaixo
            # (mas o loop começa em start_idx, então não atrapalha)

        pending_tokens, pending_idx = [], []

        t0 = time.time()
        for ex_idx, ex in enumerate(tqdm(all_test[start_idx:], desc=f"Inferindo: {nome}"), start=start_idx):
            tokens = ex["tokens"]
            k = _hash_key(tokens, model_escolhido, shot_id=split_salt)

            golds[nome][ex_idx] = [ex["ner_tags"]]

            if k in cache:
                # cache hit -> escreve e avança checkpoint
                predictions[nome][ex_idx] = [cache[k]]
                save_checkpoint(nome, ex_idx + 1)
            else:
                pending_tokens.append(tokens)
                pending_idx.append(ex_idx)

                # Quando o batch enche, consulta o LLM uma vez
                if len(pending_tokens) >= BATCH_SIZE:
                    try:
                        batch_tags = ner.tag_batch(pending_tokens)
                    except Exception as e:
                        # Fallback: processa 1 a 1; se falhar, põe 'O' e loga
                        batch_tags = []
                        for toks, idx_buf in zip(pending_tokens, pending_idx):
                            try:
                                one = ner.tag_batch([toks])[0]
                            except Exception as ee:
                                one = ["O"] * len(toks)
                                log_fail(nome, idx_buf, ee)
                            batch_tags.append(one)

                    # cacheia e escreve por índice
                    for toks, tags in zip(pending_tokens, batch_tags):
                        kk = _hash_key(toks, model_escolhido, shot_id=split_salt)
                        cache[kk] = tags
                        _append_cache(CACHE_PATH, kk, tags)

                    for idx_buf, tags in zip(pending_idx, batch_tags):
                        predictions[nome][idx_buf] = [tags]
                        save_checkpoint(nome, idx_buf + 1)  # avança sempre

                    pending_tokens, pending_idx = [], []

        # Flush final se sobrou algo
        if pending_tokens:
            try:
                batch_tags = ner.tag_batch(pending_tokens)
            except Exception as e:
                batch_tags = []
                for toks, idx_buf in zip(pending_tokens, pending_idx):
                    try:
                        one = ner.tag_batch([toks])[0]
                    except Exception as ee:
                        one = ["O"] * len(toks)
                        log_fail(nome, idx_buf, ee)
                    batch_tags.append(one)

            for toks, tags in zip(pending_tokens, batch_tags):
                kk = _hash_key(toks, model_escolhido, shot_id=split_salt)
                cache[kk] = tags
                _append_cache(CACHE_PATH, kk, tags)

            for idx_buf, tags in zip(pending_idx, batch_tags):
                predictions[nome][idx_buf] = [tags]
                save_checkpoint(nome, idx_buf + 1)

            pending_tokens, pending_idx = [], []

        dt = time.time() - t0
        print(f"[{nome}] Terminado em {dt/60:.2f} min")

    return golds, predictions

In [47]:
# golds, predictions = {}, {}
# cache = _load_cache(CACHE_PATH)
# for split, nome in zip(splits, splits_names):
#         split_base = split
#         total = 500  # ou len(split_base['test'])
#         few_shot = select_few_shot(split_base["train"], FEW_SHOT_K)
# golds, predictions = {}, {}
# cache = _load_cache(CACHE_PATH)

# split = splits[1]
# nome = splits_names[1]
# split_base = split
# total = 15

# few_shot = select_few_shot(split_base["train"], FEW_SHOT_K)

# ner = OllamaNER(
#             model=model_escolhido,
#             few_shot_text=few_shot,
#             options={
#                 "temperature": 0,
#                 "num_ctx": 768,         # ajuste isso se o few-shot for maior
#                 "num_predict": 1536,    # saída JSON pode ser grande; mantenha folga
#             },
#             keep_alive="10m",
#             json_mode=True,
#         )


# golds[nome] = [None] * total
# predictions[nome] = [None] * total

# # Config de batch (experimente 4, 8, 12 e escolha o melhor no seu hardware)
# BATCH_SIZE = 5

# # ✅ Monta a fila com cache: só manda ao LLM o que não estiver no cache
# pending_tokens = []
# pending_idx = []
# all_test = list(split_base["test"].select(range(total)))
# split_salt = f"split:{nome}"

# t0 = time.time()
# for ex_idx, ex in enumerate(tqdm(all_test, desc=f"Inferindo: {nome}")):
#     tokens = ex["tokens"]
#     k = _hash_key(tokens, model_escolhido, shot_id=split_salt)
#     golds[nome][ex_idx] = [ex["ner_tags"]]

#     if k in cache:
#         # cache hit
#         predictions[nome][ex_idx] = [cache[k]]
#     else:
#         pending_tokens.append(tokens)
#         pending_idx.append(ex_idx)

#         # Quando o batch enche, consulta o LLM uma vez
#         if len(pending_tokens) >= BATCH_SIZE:
#             try:
#                 batch_tags = ner.tag_batch(pending_tokens)
#                 ok = True
#             except Exception as e:
#                 ok = False
#                 # Fallback: processa 1 a 1; se falhar, preenche 'O' e loga
#                 batch_tags = []
#                 for toks, idx_buf in zip(pending_tokens, pending_idx):
#                     try:
#                         one = ner.tag_batch([toks])[0]
#                     except Exception as ee:
#                         one = ["O"] * len(toks)
#                         log_fail(nome, idx_buf, ee)
#                     batch_tags.append(one)

#             # cacheia e escreve por índice
#             for toks, tags in zip(pending_tokens, batch_tags):
#                 kk = _hash_key(toks, model_escolhido, shot_id=split_salt)
#                 cache[kk] = tags
#                 _append_cache(CACHE_PATH, kk, tags)

#             for idx_buf, tags in zip(pending_idx, batch_tags):
#                 predictions[nome][idx_buf] = [tags]
#                 save_checkpoint(nome, idx_buf + 1)  # <<< checkpoint avanço a cada exemplo

#             pending_tokens, pending_idx = [], []
# # Flush final se sobrou algo

# if pending_tokens:
#     try:
#         batch_tags = ner.tag_batch(pending_tokens)
#     except Exception as e:
#         batch_tags = []
#         for toks, idx_buf in zip(pending_tokens, pending_idx):
#             try:
#                 one = ner.tag_batch([toks])[0]
#             except Exception as ee:
#                 one = ["O"] * len(toks)
#                 log_fail(nome, idx_buf, ee)
#             batch_tags.append(one)

#     for toks, tags in zip(pending_tokens, batch_tags):
#         kk = _hash_key(toks, model_escolhido, shot_id=split_salt)
#         cache[kk] = tags
#         _append_cache(CACHE_PATH, kk, tags)

#     for idx_buf, tags in zip(pending_idx, batch_tags):
#         predictions[nome][idx_buf] = [tags]
#         save_checkpoint(nome, idx_buf + 1)

#     pending_tokens, pending_idx = [], []

# dt = time.time() - t0
# print(f"[{nome}] Terminado em {dt/60:.2f} min")

# t0 = time.time()
# for ex_idx, ex in enumerate(tqdm(all_test, desc=f"Inferindo: {nome}")):
#     tokens = ex["tokens"]
#     split_salt = f"split:{nome}"
#     k = _hash_key(tokens, model_escolhido, shot_id=split_salt)
#     golds[nome][ex_idx] = [ex["ner_tags"]]

#     if k in cache:
#         # cache hit
#         predictions[nome][ex_idx] = [cache[k]]
#     else:
#         pending_tokens.append(tokens)
#         pending_idx.append(ex_idx)

#         # Quando o batch enche, consulta o LLM uma vez
#         if len(pending_tokens) >= BATCH_SIZE:
#             batch_tags = ner.tag_batch(pending_tokens)
#             for toks, tags in zip(pending_tokens, batch_tags):
#                 kk = _hash_key(toks, model_escolhido, shot_id=split_salt)
#                 cache[kk] = tags
#                 _append_cache(CACHE_PATH, kk, tags)
#             # escreve nas posições corretas
#             for idx_buf, tags in zip(pending_idx, batch_tags):
#                 predictions[nome][idx_buf] = [tags]
#             pending_tokens, pending_idx = [], []
# # Flush final se sobrou algo

# if pending_tokens:
#     batch_tags = ner.tag_batch(pending_tokens)

#     for toks, tags in zip(pending_tokens, batch_tags):
#         kk = _hash_key(toks, model_escolhido, shot_id=split_salt)
#         cache[kk] = tags
#         _append_cache(CACHE_PATH, kk, tags)

#     for idx_buf, tags in zip(pending_idx, batch_tags):
#         predictions[nome][idx_buf] = [tags]

#     pending_tokens, pending_idx = [], []

# dt = time.time() - t0
# print(f"[{nome}] Terminado em {dt/60:.2f} min")

In [48]:
golds, predictions = run_fast_inference(
    splits=splits,                      # já existentes no seu notebook
    splits_names=splits_names,          # idem
    model_escolhido=model_escolhido,
    FEW_SHOT_K=FEW_SHOT_K,
    select_few_shot=select_few_shot,
    SEED_GLOBAL=SEED_GLOBAL,
)

Inferindo: standard: 0it [00:00, ?it/s]


[standard] Terminado em 0.00 min


Inferindo: heur_len: 100%|██████████| 65/65 [5:59:08<00:00, 331.51s/it]  


[heur_len] Terminado em 366.94 min


Inferindo: heur_rare:  32%|███▏      | 159/500 [21:54:56<47:00:06, 496.21s/it] 


KeyboardInterrupt: 

In [ ]:
predictions

{'standard': [[['B-ORGANIC',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O']],
  [['B-OCCURRENCE',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O']],
  [['O',
    'B-ORG',
    'I-ORG',
    'O',
    'B-LOC',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'B-PER',
    'I-PER',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',


In [ ]:
golds

{'standard': [[['O',
    'O',
    'O',
    'B-baciaSedimentar',
    'I-baciaSedimentar',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O']],
  [['O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'B-idade',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O']],
  [['O',
    'O',
    'O',
    'B-procedimentoMetodologico',
    'I-procedimentoMetodologico',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    'O',
    '

In [ ]:
def _flatten_token_level(golds_split, preds_split):
    """
    golds_split:  ex.: [ [tags], [tags], ... ] mas no seu caso vem como [ [ [tags] ], ... ]
    preds_split:  mesmo formato
    retorna y_true_flat, y_pred_flat (listas de strings)
    """
    y_true_flat, y_pred_flat = [], []
    for g, p in zip(golds_split, preds_split):
        if g is None or p is None:
            continue
        # no seu dicionário: cada item é [tags]; então g[0] e p[0]
        g_seq = g[0]  # seu formato: [ [tags] ]
        p_seq = p[0]
        # sanitiza antes de alinhar
        g_seq = _coerce_seq(g_seq)
        p_seq = _coerce_seq(p_seq)
        n = min(len(g_seq), len(p_seq))
        if n <= 0:
            continue
        y_true_flat.extend(g_seq[:n])
        y_pred_flat.extend(p_seq[:n])
    return y_true_flat, y_pred_flat

def compute_token_metrics_for_split(golds_split, preds_split, labels=None):
    y_true, y_pred = _flatten_token_level(golds_split, preds_split)

    # define labels se não vierem
    if labels is None:
        labels = sorted({*y_true, *y_pred})

    # relatório sklearn (NÃO seqeval)
    rep = sk_classification_report(y_true, y_pred, labels=labels,
                                   output_dict=True, zero_division=0)
    p_micro, r_micro, f_micro, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average='micro', zero_division=0
    )
    rep.setdefault("micro avg", {})
    rep["micro avg"].update({"precision": p_micro, "recall": r_micro, "f1-score": f_micro})

    metrics = {
        "precision_micro":    rep["micro avg"]["precision"],
        "recall_micro":       rep["micro avg"]["recall"],
        "f1_micro":           rep["micro avg"]["f1-score"],
        "precision_macro":    rep["macro avg"]["precision"],
        "recall_macro":       rep["macro avg"]["recall"],
        "f1_macro":           rep["macro avg"]["f1-score"],
        "precision_weighted": rep["weighted avg"]["precision"],
        "recall_weighted":    rep["weighted avg"]["recall"],
        "f1_weighted":        rep["weighted avg"]["f1-score"],
        "accuracy":           accuracy_score(y_true, y_pred),
        "support":            len(y_true),
        "num_labels":         len(labels),
    }
    return metrics, rep, (y_true, y_pred)


In [ ]:
for nome in predictions.keys():
    print(nome)

standard
heur_len
heur_rare
adversarial


In [ ]:
from sklearn.metrics import (
    classification_report as sk_classification_report,
    accuracy_score,
    precision_recall_fscore_support,
)

In [ ]:
_IOB_RE = re.compile(r"^[BIO]-")

def _coerce_tag_value_from_dict(d: dict) -> str:
    for key in ("tag", "label", "iob", "iob_tag", "ner", "entity", "type"):
        v = d.get(key)
        if isinstance(v, str):
            v = v.strip()
            if v == "O" or _IOB_RE.match(v):
                return v
    return "O"

In [ ]:
def _coerce_seq(seq):
    """Converte uma sequência possivelmente heterogênea (str/dict/outros) -> lista de strings IOB/O."""
    out = []
    for t in seq:
        if isinstance(t, str):
            out.append(t if (t == "O" or _IOB_RE.match(t)) else "O")
        elif isinstance(t, dict):
            out.append(_coerce_tag_value_from_dict(t))
        else:
            out.append("O")
    return out

In [ ]:
metrics_by_split = {}
reports_by_split = {}
y_true_all, y_pred_all = [], []

# se quiser uma lista de labels fixa (estável entre splits), passe em labels=
for nome in predictions.keys():
    m, rep, (y_true, y_pred) = compute_token_metrics_for_split(
        golds[nome], predictions[nome]
    )
    metrics_by_split[nome] = m
    reports_by_split[nome] = rep

In [ ]:
metrics_by_split

{'standard': {'precision_micro': 0.8747577158192933,
  'recall_micro': 0.8747577158192933,
  'f1_micro': 0.8747577158192933,
  'precision_macro': 0.0023856789324259594,
  'recall_macro': 0.0024279237300059426,
  'f1_macro': 0.0024066159583208736,
  'precision_weighted': 0.8595373189923204,
  'recall_weighted': 0.8747577158192933,
  'f1_weighted': 0.8670807293233525,
  'accuracy': 0.8747577158192933,
  'support': 13414,
  'num_labels': 389},
 'heur_len': {'precision_micro': 0.8539738392105867,
  'recall_micro': 0.8539738392105867,
  'f1_micro': 0.8539738392105867,
  'precision_macro': 0.0016925651101734954,
  'recall_macro': 0.0015685539469334445,
  'f1_macro': 0.0016010003843055188,
  'precision_weighted': 0.8636201075219491,
  'recall_weighted': 0.8539738392105867,
  'f1_weighted': 0.8587191960226679,
  'accuracy': 0.8539738392105867,
  'support': 13073,
  'num_labels': 603},
 'heur_rare': {'precision_micro': 0.8822186873227748,
  'recall_micro': 0.8822186873227748,
  'f1_micro': 0.88

In [ ]:
SAFE_MODEL = re.sub(r'[^A-Za-z0-9._-]+', '_', MODEL_KEY)  # "gemma2:9b" -> "gemma2_9b"
BASE_DIR   = Path("results") / SAFE_MODEL
DIR_METRICS = BASE_DIR / "metrics"
DIR_REPORTS = BASE_DIR / "reports"
DIR_PAIRS   = BASE_DIR / "pairs"      # gold/pred pareados por linha (jsonl)
DIR_RAW     = BASE_DIR / "raw"        # dumps completos (golds/predictions)

In [ ]:
RUN_TAG = time.strftime("%Y%m%d-%H%M%S")

In [ ]:
for d in (DIR_METRICS, DIR_REPORTS, DIR_PAIRS, DIR_RAW):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
for nome in predictions.keys():
    # métricas token-level agregadas do split
    with open(DIR_METRICS / f"{nome}.metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics_by_split[nome], f, ensure_ascii=False, indent=2)

    # classification_report (dict) do split
    with open(DIR_REPORTS / f"{nome}.report.json", "w", encoding="utf-8") as f:
        json.dump(reports_by_split[nome], f, ensure_ascii=False, indent=2)

    # pares gold/pred por exemplo (jsonl: 1 linha = 1 ex)
    pairs_path = DIR_PAIRS / f"{nome}.jsonl"
    with open(pairs_path, "w", encoding="utf-8") as f:
        for i, (g, p) in enumerate(zip(golds[nome], predictions[nome])):
            if g is None or p is None:
                continue
            rec = {"idx": i, "gold": g[0], "pred": p[0]}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# ---- salvar agregados (opcional) ----
# dicionários completos (podem ficar grandes)
with open(DIR_RAW / f"golds.{RUN_TAG}.json", "w", encoding="utf-8") as f:
    json.dump(golds, f, ensure_ascii=False)
with open(DIR_RAW / f"predictions.{RUN_TAG}.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False)


In [ ]:
# report = {}
# f1_macro = {}
# precision = {}
# recall = {}
# metrics_all = {}

# bads_sets = {}

# for name, predidction in predictions.items():
#     preds = []
#     print(predidction)
#     for tags_prediction in predidction:
#         try:
#             preds.append(json.loads(tags_prediction)['tags']) 
#         except:
#             try:
#                 preds.append(json.loads(tags_prediction[0])['tags']) 
#             except:
#                 try:
#                     preds.append(json.loads(tags_prediction[0][0])['tags']) 
#                 except:
#                     preds.append(['O'])
          
#     new_preds = []
#     for pred_token in preds:
#         if type(pred_token) == str:
#             new_preds.append(pred_token)
#         else:
#             tokens_ajustados = []  
#             for token in pred_token:
#                 try:
#                     if int(_).is_integer():
#                         tokens_ajustados.extend([token_ajustado])
#                     else:
#                         tokens_ajustados.extend([token])
#                 except:
#                     try:
#                         token_ajustado = token['tag']
#                         tokens_ajustados.extend([token_ajustado])
#                     except:
#                         token_ajustado = token
#                         tokens_ajustados.extend([token_ajustado])
#             new_preds.append(tokens_ajustados)

        
#     if type(golds[name][0][0]) == list:
#         actual_golds = [gold[0] for gold in golds[name]]
#         print('ok')
#     else:
#         actual_golds = golds[name]
#     actual_preds = new_preds
#     print(new_preds)
#     # clean_golds, cleaned, clean_preds = [], [], []

#     # for i, (g, p) in enumerate(zip(actual_golds, actual_preds)):
#     #     if type(g[0][0]) == list:
#     #         g_ = g[0]
#     #         print(len(g_), len(p))
#     #         if len(p) < len(g_):
#     #             print('limpou preds'); print(i)
#     #             cleaned.append(i)
#     #             continue  # pula a adição
#     #         elif len(p) > len(g_):
#     #             print('limpou golds'); print(i)
#     #             cleaned.append(i)
#     #             continue  # pula a adição
#     #     else:
#     #         print(len(g), len(p))
#     #         if len(p) < len(g):
#     #             print('limpou preds'); print(i)
#     #             cleaned.append(i)
#     #             continue  # pula a adição
#     #         elif len(p) > len(g):
#     #             print('limpou golds'); print(i)
#     #             cleaned.append(i)
#     #             continue  # pula a adição

#     clean_golds, clean_preds = [], []
#     for g, p in zip(actual_golds, actual_preds):
#         # se o modelo gerar menos/more tags que tokens, ajustamos:
#         if type(g[0][0]) == list:
#             g_ = g[0]
#             if len(p) < len(g_):
#                 print('limpou preds')
#                 p = p + ["O"] * (len(g_) - len(p))          # completa com O
#             elif len(p) > len(g_):
#                 print('limpou golds')
#                 p = p[:len(g_)]                             # descarta excedente
#             clean_golds.append(g_)
#             clean_preds.append(p)
#         else:
#             if len(p) < len(g):
#                 print('limpou preds')
#                 p = p + ["O"] * (len(g) - len(p))          # completa com O
#             elif len(p) > len(g):
#                 print('limpou golds')
#                 p = p[:len(g)]                             # descarta excedente
#             clean_golds.append(g)
#             clean_preds.append(p)

#     # bad = set(cleaned)

#     # bads_sets[name] = bad

#     # clean_golds[:] = [g for i, g in enumerate(actual_golds) if i not in bad]
#     # clean_preds[:] = [p for i, p in enumerate(actual_preds) if i not in bad]

#     # print(f"{name}: {len(clean_golds)} exemplos válidos ({len(bad)} removidos)")
#     # print(clean_golds, len(clean_golds))
#     # print(clean_preds, len(clean_preds))
#     if len(clean_golds) != 0 and len(clean_preds) != 0:
#         report[name] = classification_report(clean_golds, clean_preds, output_dict=True)
#         f1_macro[name] = f1_score(clean_golds, clean_preds, average="macro")
#         precision[name] = precision_score(clean_golds, clean_preds, average="macro")
#         recall[name] = recall_score(clean_golds, clean_preds, average="macro")
#         rep = classification_report(clean_golds, clean_preds, scheme=IOB2, zero_division=0, output_dict=True)

#         metrics = {
#             "precision_micro":   rep["micro avg"]["precision"],
#             "recall_micro":      rep["micro avg"]["recall"],
#             "f1_micro":          rep["micro avg"]["f1-score"],
#             "precision_macro":   rep["macro avg"]["precision"],
#             "recall_macro":      rep["macro avg"]["recall"],
#             "f1_macro":          rep["macro avg"]["f1-score"],
#             "precision_weighted":rep["weighted avg"]["precision"],
#             "recall_weighted":   rep["weighted avg"]["recall"],
#             "f1_weighted":       rep["weighted avg"]["f1-score"],
#             "accuracy":          accuracy_score(clean_golds, clean_preds),
#         }

#         metrics_all[name] = metrics

In [ ]:
# precision_micro = []
# recall_micro = []
# f1_micro = []
# f1_macro = []
# precision_macro = []
# recall_macro = []
# f1_weighted = []
# precision_weighted = []
# recall_weighted = []
# accuracy = []
# name = []

# for keys,value in metrics_all.items():
#     name.append(keys)
#     precision_micro.append(value['precision_micro'])
#     recall_micro.append(value['recall_micro'])
#     f1_micro.append(value['f1_micro'])
#     precision_macro.append(value['precision_macro'])
#     recall_macro.append(value['recall_macro'])
#     f1_macro.append(value['f1_macro'])
#     precision_weighted.append(value['precision_weighted'])
#     recall_weighted.append(value['recall_weighted'])
#     f1_weighted.append(value['f1_weighted'])
#     accuracy.append(value['accuracy'])

# df_results = pd.DataFrame({'name':name,
#                            'precision_micro':precision_micro,
#                            'recall_micro':recall_micro,
#                            'f1_micro':f1_micro,
#                            'precision_macro':precision_macro,
#                            'recall_macro':recall_macro,
#                            'f1_macro':f1_macro,
#                            'precision_weighted':precision_weighted,
#                            'recall_weighted':recall_weighted,
#                            'f1_weighted':f1_weighted,
#                            'accuracy':accuracy})


In [ ]:
# df_results.to_csv(f'metricas_{model_escolhido.replace(':', '')}_exclusao.csv', index=False)

In [ ]:
# report = {}
# f1_macro = {}
# precision = {}
# recall = {}
# metrics_all2 = {}

# bads_sets = {}

# for name, predidction in predictions.items():
#     preds = []
#     print(predidction)
#     for tags_prediction in predidction:
#         try:
#             preds.append(json.loads(tags_prediction)['tags']) 
#         except:
#             try:
#                 preds.append(json.loads(tags_prediction[0])['tags']) 
#             except:
#                 try:
#                     preds.append(json.loads(tags_prediction[0][0])['tags']) 
#                 except:
#                     preds.append(['O'])
          
#     new_preds = []
#     for pred_token in preds:
#         if type(pred_token) == str:
#             new_preds.append(pred_token)
#         else:
#             tokens_ajustados = []  
#             for token in pred_token:
#                 try:
#                     if int(_).is_integer():
#                         tokens_ajustados.extend([token_ajustado])
#                     else:
#                         tokens_ajustados.extend([token])
#                 except:
#                     try:
#                         token_ajustado = token['tag']
#                         tokens_ajustados.extend([token_ajustado])
#                     except:
#                         token_ajustado = token
#                         tokens_ajustados.extend([token_ajustado])
#             new_preds.append(tokens_ajustados)

        
#     if type(golds[name][0][0]) == list:
#         actual_golds = [gold[0] for gold in golds[name]]
#         print('ok')
#     else:
#         actual_golds = golds[name]
#     actual_preds = new_preds
#     print(new_preds)
#     clean_golds, cleaned, clean_preds = [], [], []

#     for i, (g, p) in enumerate(zip(actual_golds, actual_preds)):
#         if type(g[0][0]) == list:
#             g_ = g[0]
#             print(len(g_), len(p))
#             if len(p) < len(g_):
#                 print('limpou preds'); print(i)
#                 cleaned.append(i)
#                 continue  # pula a adição
#             elif len(p) > len(g_):
#                 print('limpou golds'); print(i)
#                 cleaned.append(i)
#                 continue  # pula a adição
#         else:
#             print(len(g), len(p))
#             if len(p) < len(g):
#                 print('limpou preds'); print(i)
#                 cleaned.append(i)
#                 continue  # pula a adição
#             elif len(p) > len(g):
#                 print('limpou golds'); print(i)
#                 cleaned.append(i)
#                 continue  # pula a adição

#     bad = set(cleaned)

#     bads_sets[name] = bad

#     clean_golds[:] = [g for i, g in enumerate(actual_golds) if i not in bad]
#     clean_preds[:] = [p for i, p in enumerate(actual_preds) if i not in bad]


#     if len(clean_golds) != 0 and len(clean_preds) != 0:
#         report[name] = classification_report(clean_golds, clean_preds, output_dict=True)
#         f1_macro[name] = f1_score(clean_golds, clean_preds, average="macro")
#         precision[name] = precision_score(clean_golds, clean_preds, average="macro")
#         recall[name] = recall_score(clean_golds, clean_preds, average="macro")

#         rep = classification_report(clean_golds, clean_preds, scheme=IOB2, zero_division=0, output_dict=True)
        
#         metrics2 = {
#             "precision_micro":   rep["micro avg"]["precision"],
#             "recall_micro":      rep["micro avg"]["recall"],
#             "f1_micro":          rep["micro avg"]["f1-score"],
#             "precision_macro":   rep["macro avg"]["precision"],
#             "recall_macro":      rep["macro avg"]["recall"],
#             "f1_macro":          rep["macro avg"]["f1-score"],
#             "precision_weighted":rep["weighted avg"]["precision"],
#             "recall_weighted":   rep["weighted avg"]["recall"],
#             "f1_weighted":       rep["weighted avg"]["f1-score"],
#             "accuracy":          accuracy_score(clean_golds, clean_preds),
#         }

#         metrics_all2[name] = metrics2

In [ ]:
# precision_micro2 = []
# recall_micro2 = []
# f1_micro2 = []
# f1_macro2 = []
# precision_macro2 = []
# recall_macro2 = []
# f1_weighted2 = []
# precision_weighted2 = []
# recall_weighted2 = []
# accuracy2 = []
# name2 = []

# for keys,value in metrics_all2.items():
#     name2.append(keys)
#     precision_micro2.append(value['precision_micro'])
#     recall_micro2.append(value['recall_micro'])
#     f1_micro2.append(value['f1_micro'])
#     precision_macro2.append(value['precision_macro'])
#     recall_macro2.append(value['recall_macro'])
#     f1_macro2.append(value['f1_macro'])
#     precision_weighted2.append(value['precision_weighted'])
#     recall_weighted2.append(value['recall_weighted'])
#     f1_weighted2.append(value['f1_weighted'])
#     accuracy2.append(value['accuracy'])

# df_results2 = pd.DataFrame({'name':name2,
#                            'precision_micro':precision_micro2,
#                            'recall_micro':recall_micro2,
#                            'f1_micro':f1_micro2,
#                            'precision_macro':precision_macro2,
#                            'recall_macro':recall_macro2,
#                            'f1_macro':f1_macro2,
#                            'precision_weighted':precision_weighted2,
#                            'recall_weighted':recall_weighted2,
#                            'f1_weighted':f1_weighted2,
#                            'accuracy_score':accuracy2})


In [ ]:
# df_results2.to_csv(f'metricas_{model_escolhido.replace(':', '')}_complemento.csv', index=False)

In [ ]:
# df_results2